In [ ]:
# =============================================================================
# CROSS-ENCODER — EXPORT for v4 (run in the Step 2 v4 notebook that GATHERED all
# countries, after its full run). Writes only the v4 band pairs that the cross-encoder
# has NOT scored yet (not in the v3 ce_test.parquet) -> ce_band_test_v4.parquet
# INPUTS  ce_config_v4.json (CE-1c) and the v3 ce_test.parquet (CE-2), attached.
# =============================================================================
import json
import numpy as np, pandas as pd
from pathlib import Path

W, I = Path("/kaggle/working"), Path("/kaggle/input")
cf = [f for f in list(I.rglob("ce_config_v4.json")) + list(W.rglob("ce_config_v4.json"))]
assert cf, "ce_config_v4.json (CE-1c output) not found: attach it"
BAND = tuple(json.load(open(cf[0]))["band"])
done_f = list(I.rglob("ce_test.parquet")) + list(W.glob("ce_test.parquet"))
done = (pd.read_parquet(done_f[0], columns=["s1", "cand"]).set_index(["s1", "cand"]).index
        if done_f else pd.MultiIndex.from_arrays([[], []]))
print(f"band {BAND} | already scored by the cross-encoder: {len(done):,} pairs")
PARTS, CKPT = W / "stage2_v4_output" / "_parts", W / "ckpt_test"
assert PARTS.exists(), "stage2_v4_output/_parts not found: gather + run Step 2 v4 first"
out = []
for part in sorted(PARTS.glob("*.npz")):
    c = part.stem
    z = np.load(part)
    qids = np.load(CKPT / c / "qids.npy", allow_pickle=True)
    pids = np.load(CKPT / c / "pids.npy", allow_pickle=True)
    b = (z["prob"] >= BAND[0]) & (z["prob"] < BAND[1])          # v4 used margin 20: no clamp needed
    d = pd.DataFrame({"country": c, "s1": qids[z["q"][b]], "cand": pids[z["p"][b]], "prob": z["prob"][b]})
    new = ~pd.MultiIndex.from_frame(d[["s1", "cand"]]).isin(done)
    out.append(d[new])
    print(f"{c:<7} kept {len(z['q']):>12,} | band {int(b.sum()):>10,} | already scored {int((~new).sum()):>10,} "
          f"| NEW {int(new.sum()):>10,}")
out = pd.concat(out, ignore_index=True)
out.to_parquet(W / "ce_band_test_v4.parquet", index=False)
print(f"\nsaved {W / 'ce_band_test_v4.parquet'}: {len(out):,} NEW pairs for CE-2")